In [2]:
import glob, os
from collections import defaultdict
import pandas as pd

folder = "."  # CSV가 있는 폴더 (하위 폴더까지 탐색)
files = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))

groups = defaultdict(list)
for f in files:
    try:
        df = pd.read_csv(f)
    except Exception as e:
        print("읽기 실패:", os.path.basename(f), "|", e)
        continue
    groups[tuple(df.columns)].append((f, df))

out = [f"CSV 파일 총 {len(files)}개, 컬럼 구성 그룹 {len(groups)}개\n"]
for i, (cols, items) in enumerate(groups.items(), 1):
    rows = [d.shape[0] for _, d in items]
    first_df = items[0][1]
    out.append(f"=== 그룹 {i}: 파일 {len(items)}개 / 컬럼 {len(cols)}개 ===")
    out.append(f"예시 파일: {[os.path.basename(f)[:30] for f, _ in items[:2]]}")
    out.append(f"행 수 최소/최대/합계: {min(rows)} / {max(rows)} / {sum(rows)}")
    out.append(f"dtypes:\n{first_df.dtypes.to_string()}")
    out.append(f"head(3):\n{first_df.head(3).to_string()}")
    na = first_df.isna().sum()
    out.append(f"결측치(첫 파일 기준):\n{na[na > 0].to_string() if (na > 0).any() else '없음'}")
    for c in ("ADD", "resp24_50"):  # 요약표에만 있는 컬럼
        if c in first_df.columns:
            out.append(f"{c} 분포:\n{first_df[c].value_counts(dropna=False).to_string()}")
    out.append("")

text = "\n".join(out)
print(text)
with open("summary.txt", "w", encoding="utf-8") as fh:
    fh.write(text)

CSV 파일 총 108개, 컬럼 구성 그룹 8개

=== 그룹 1: 파일 2개 / 컬럼 5개 ===
예시 파일: ['Graham_BOWL_cMMMP_5min_boat_wi', 'Graham_BOWL_cMMMP_5min_boat_wi']
행 수 최소/최대/합계: 20213 / 121805 / 142018
dtypes:
MMSI.x                 int64
DATE.TIME.x              str
Deployment.number        str
BOAT.LATITUDE        float64
BOAT.LONGITUDE       float64
head(3):
      MMSI.x       DATE.TIME.x Deployment.number  BOAT.LATITUDE  BOAT.LONGITUDE
0  311031800  19/03/2017 03:14               739       58.28301        -2.83774
1  311031800  19/03/2017 03:14               722       58.28301        -2.83774
2  311031800  19/03/2017 03:19               722       58.28300        -2.83774
결측치(첫 파일 기준):
없음

=== 그룹 2: 파일 100개 / 컬럼 5개 ===
예시 파일: ['712 146 2017 02 17 POD711 file', '713 145 2017 02 17 POD1022 V2 ']
행 수 최소/최대/합계: 80639 / 298381 / 17794041
dtypes:
File           str
ChunkEnd       str
Nfiltered    int64
Nall         int64
MinsOn       int64
head(3):
                                      File         ChunkEnd  Nfiltered  

In [3]:
df = pd.read_csv("Graham_BOWL_cMMMP_Porpoise_responses_to_construction_data_2019-05-01.csv")
print(df.groupby("turbine").size().describe())
t = df.groupby("turbine").agg(ADD=("ADD", lambda s: s.nunique()),
        add_first=("ADD", "first"), order=("piling_order", "first"))
print(t["ADD"].value_counts())            # 1이면 터빈 안에서 ADD가 일정
print(t.groupby("add_first")["order"].describe())
print(df[["distance","Unweighted_SS_SEL","NOAA_SS_SEL","Southall_SS_SEL","Aud_SS_SEL"]].corr().round(2))
print(df.groupby("ADD")["resp24_50"].agg(["mean","count"]))

count    18.000000
mean     40.111111
std      12.494182
min       9.000000
25%      44.000000
50%      45.000000
75%      47.000000
max      47.000000
dtype: float64
ADD
1    18
Name: count, dtype: int64
           count     mean        std   min   25%   50%   75%   max
add_first                                                         
N            2.0  60.0000   1.414214  59.0  59.5  60.0  60.5  61.0
Y           16.0  43.8125  35.318019   1.0   9.5  39.5  80.0  86.0
                   distance  Unweighted_SS_SEL  NOAA_SS_SEL  Southall_SS_SEL  \
distance               1.00              -0.94        -0.94            -0.93   
Unweighted_SS_SEL     -0.94               1.00         0.99             1.00   
NOAA_SS_SEL           -0.94               0.99         1.00             0.99   
Southall_SS_SEL       -0.93               1.00         0.99             1.00   
Aud_SS_SEL            -0.92               0.99         1.00             0.99   

                   Aud_SS_SEL  
distance      

In [4]:
t = df.groupby("turbine").agg(n=("dep_no","size"), order=("piling_order","first"),
        add=("ADD","first"), dur=("duration","first"), resp=("resp24_50","mean"))
print(t.sort_values("order").round(2).to_string())
print(df["dep_no"].nunique(), df.groupby("dep_no").size().describe())

          n  order add   dur  resp
turbine                           
G7       45      1   Y  7.10  0.44
F8       45      2   Y  5.78  0.39
E2       45      4   Y  4.67  0.20
J5       45      8   Y  7.05  0.43
G5       44     10   Y  5.74  0.53
F5       35     12   Y  4.64  0.56
J8        9     23   Y  3.27  0.22
E4        9     32   Y  7.96  0.11
L8       26     47   Y  5.48  0.19
D11      47     59   N  6.72  0.26
G12      47     61   N  3.52  0.21
F11      47     70   Y  4.47  0.21
E11      47     75   Y  6.18  0.22
H13      47     79   Y  3.44  0.26
H10      47     83   Y  4.62  0.26
G9       47     84   Y  5.29  0.13
H9       46     85   Y  4.01  0.26
J10      44     86   Y  4.45  0.21
100 count    100.000000
mean       7.220000
std        2.236429
min        3.000000
25%        6.000000
50%        6.000000
75%        9.000000
max       10.000000
dtype: float64
